In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import recall_score, f1_score, roc_auc_score

DATA_PATH = os.path.join('..', 'vr_legibility_train.csv')

c:\Users\psgve\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 데이터 로드 및 전처리
df = pd.read_csv(DATA_PATH)[["Yaw", "Pitch", "FontSize", "Label"]]

# 특성(X)과 타겟(y) 분리
X = df.drop('Label', axis=1)
y = df['Label']

# 2. 학습/테스트 데이터 분할 (Stratified Sampling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# 교차검증 설정 (StratifiedKFold 적용)
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [3]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
preprocessor = make_column_transformer(
    (MinMaxScaler(), ['Yaw', 'Pitch']),
    (make_pipeline(FunctionTransformer(np.log1p), MinMaxScaler()), ['FontSize'])
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))
])

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial, X_train_val, y_train_val, pipeline, inner_cv):
    # 1. 하이퍼파라미터 탐색 범위 정의
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet', None])
    
    # penalty에 따른 solver 및 추가 파라미터 제약 조건 설정
    if penalty == 'l1':
        solver = trial.suggest_categorical('solver_l1', ['liblinear', 'saga'])
        l1_ratio = None
    elif penalty == 'l2':
        solver = trial.suggest_categorical('solver_l2', ['lbfgs', 'newton-cg', 'sag', 'saga'])
        l1_ratio = None
    elif penalty == 'elasticnet':
        solver = 'saga'  # elasticnet은 saga만 지원
        l1_ratio = trial.suggest_float('l1_ratio', 0.001, 0.1)
    else:  # penalty is None ('none')
        solver = trial.suggest_categorical('solver_none', ['lbfgs', 'newton-cg', 'sag', 'saga'])
        l1_ratio = None

    # 규제 강도 C (로그 스케일 탐색)
    C = trial.suggest_float('C', 1e-8, 1.0, log=True) if penalty != 'none' else 1.0

    # 2. 모델 설정 (기존 파이프라인 활용)
    # solver_l1, solver_l2 등으로 이름이 나뉜 것을 하나로 통합
    actual_solver = solver 
    
    actual_penalty = None if penalty == 'none' else penalty
    params = {
        'lr__penalty': penalty,
        'lr__C': C,
        'lr__solver': actual_solver,
        'lr__max_iter': 3000,
        'lr__class_weight': 'balanced',
        'lr__random_state': 42
    }
    
    if penalty == 'elasticnet':
        params['lr__l1_ratio'] = l1_ratio

    # 파이프라인에 파라미터 적용
    pipeline.set_params(**params)

    # 3. 교차 검증 점수 계산 (내부 CV)
    # scoring='f1'을 사용하여 평균 점수 반환
    score = cross_val_score(pipeline, X_train_val, y_train_val, cv=inner_cv, scoring='f1', n_jobs=-1)
    
    return score.mean()


In [8]:
import warnings
warnings.filterwarnings('ignore') # 경고 메시지 무시 

# (선택) Optuna가 매 시도마다 출력하는 로그가 너무 길어지는 것을 방지
optuna.logging.set_verbosity(optuna.logging.WARNING) 

f1_scores = []
rec_scores = []
auc_scores = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_train, y_train)):
    print(f"\n--- Outer Fold {fold_idx + 1} / 10 ---")
    X_train_val, X_test_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_val, y_test_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # 내부 베이지안 서치로 최적 하이퍼파라미터 탐색
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train_val, y_train_val, pipeline, inner_cv), n_trials=50)

    best_params_inner = study.best_params
    best_score_inner = study.best_value

    print("-" * 40)
    print(f"Optuna Best Params: {best_params_inner}")
    print(f"Best Inner CV F1:  {best_score_inner:.4f}")

    final_best_params = {
        'lr__penalty': best_params_inner['penalty'],
        'lr__C': best_params_inner.get('C', 1.0),
        'lr__solver': best_params_inner.get('solver_l1') or \
                      best_params_inner.get('solver_l2') or \
                      best_params_inner.get('solver_none') or 'saga',
    }
    if 'l1_ratio' in best_params_inner:
        final_best_params['lr__l1_ratio'] = best_params_inner['l1_ratio']
    
    # 변환된 파라미터로 모델 세팅!
    best_model_inner = pipeline.set_params(**final_best_params)
    best_model_inner.fit(X_train_val, y_train_val)

    # 검증 데이터 평가
    y_pred = best_model_inner.predict(X_test_val)
    y_score = best_model_inner.decision_function(X_test_val)

    f1 = f1_score(y_test_val, y_pred)
    rec = recall_score(y_test_val, y_pred)
    roc_auc = roc_auc_score(y_test_val, y_score)

    # 최고성능 하이퍼파라미터로 갱신 (저장할 때도 변환된 파라미터 저장)
    if fold_idx == 0:
        best_param = final_best_params
        best_f1 = f1
    elif f1 > best_f1:
        best_param = final_best_params
        best_f1 = f1

    f1_scores.append(f1)
    rec_scores.append(rec)
    auc_scores.append(roc_auc)
    print(f"\n★ Outer fold Score:: F1 Score: {f1:.4f}, Recall: {rec:.4f}, ROC AUC: {roc_auc:.4f}")

print("\n"+"-" * 10 + " FINAL RESULTS " + "-" * 10)
print(f"최종 10-Fold 평균 F1 Score: {np.mean(f1_scores):.4f} (± {np.std(f1_scores):.4f})")
print(f"최종 10-Fold 평균 Recall: {np.mean(rec_scores):.4f} (± {np.std(rec_scores):.4f})")
print(f"최종 10-Fold 평균 ROC AUC: {np.mean(auc_scores):.4f} (± {np.std(auc_scores):.4f})")
print(f"최고 F1 하이퍼파라미터: {best_param}")


--- Outer Fold 1 / 10 ---
----------------------------------------
Optuna Best Params: {'penalty': 'l1', 'solver_l1': 'saga', 'C': 0.6362895240674212}
Best Inner CV F1:  0.7152

★ Outer fold Score:: F1 Score: 0.7158, Recall: 0.6992, ROC AUC: 0.7460

--- Outer Fold 2 / 10 ---
----------------------------------------
Optuna Best Params: {'penalty': 'elasticnet', 'l1_ratio': 0.010107088673939273, 'C': 0.001195346653766435}
Best Inner CV F1:  0.7199

★ Outer fold Score:: F1 Score: 0.6697, Recall: 0.6314, ROC AUC: 0.7255

--- Outer Fold 3 / 10 ---
----------------------------------------
Optuna Best Params: {'penalty': 'elasticnet', 'l1_ratio': 0.0786881390191575, 'C': 0.014549401372285121}
Best Inner CV F1:  0.7118

★ Outer fold Score:: F1 Score: 0.7359, Recall: 0.7203, ROC AUC: 0.7866

--- Outer Fold 4 / 10 ---
----------------------------------------
Optuna Best Params: {'penalty': 'l1', 'solver_l1': 'saga', 'C': 0.18181445968939294}
Best Inner CV F1:  0.7190

★ Outer fold Score:: F1 Sc

In [9]:
# 5. 최종 모델 학습 및 저장
import joblib

print(" [최종 모델 학습 시작]")
print(X.columns)
print("=========================================")
    
final_model = pipeline.set_params(**best_param)
final_model.fit(X_train, y_train) 

# 검증 데이터 평가
final_pred = final_model.predict(X_test)
final_roc_auc = final_model.decision_function(X_test)

final_f1 = f1_score(y_test, final_pred)
final_rec = recall_score(y_test, final_pred)
roc_auc = roc_auc_score(y_test, final_roc_auc)
print(f"최종 모델 F1 Score on Test Set: {final_f1:.4f}")
print(f"최종 모델 Recall on Test Set: {final_rec:.4f}")
print(f"최종 모델 AUC Score on Test Set: {roc_auc:.4f}")

# 학습이 완료된 final_model 객체를 하드디스크에 파일로 저장합니다.
joblib.dump(final_model, os.path.join('..', 'vr_legibility_LogReg_model.pkl'))
print("✅ 최종 모델이 'vr_legibility_LogReg_model.pkl' 파일로 저장되었습니다!")

 [최종 모델 학습 시작]
Index(['Yaw', 'Pitch', 'FontSize'], dtype='object')
최종 모델 F1 Score on Test Set: 0.7439
최종 모델 Recall on Test Set: 0.7595
최종 모델 AUC Score on Test Set: 0.7695
✅ 최종 모델이 'vr_legibility_LogReg_model.pkl' 파일로 저장되었습니다!


In [ ]:
# 6. 모델 배포 후 예측 (새로운 데이터에 대한 예측)
import joblib
import pandas as pd
import os
from sklearn.metrics import recall_score, f1_score, roc_auc_score
# 1. 저장해둔 모델 불러오기 
deployed_model = joblib.load(os.path.join('..', 'vr_legibility_log_reg_model.pkl'))

# 2. 새 데이터 불러오기
"""
new_data = pd.DataFrame({
    'size': [180.0, 160.0],
    'yaw': [1250.5, -450.2],
    'pitch': [45.2, 210.8],
    "label" :[1, 1]
})
"""
new_data = pd.read_csv(os.path.join('..', 'vr_legibility_test.csv'))[["Yaw", "Pitch", "FontSize", "Label"]]

X_new = new_data[["Yaw", "Pitch", "FontSize"]]
y_true = new_data["Label"]

# 3. 새로운 데이터 예측
predictions = deployed_model.predict(X_new)
score = deployed_model.decision_function(X_new)
print("새로운 데이터 예측 완료! (총 {}개)".format(len(predictions)))

rec = recall_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
auc = roc_auc_score(y_true, score)
print("\n" + "-" * 40)
print(f"새로운 데이터에 대한 Recall: {rec:.4f}")
print(f"새로운 데이터에 대한 F1 Score: {f1:.4f}")
print(f"새로운 데이터에 대한 ROC-AUC: {auc:.4f}")